# Health Vault Document Processing — Granular 14-Stage Benchmark & Debugger

This interactive notebook benchmarks and traces **one selected medical document at a time** through the exact 14 production processing stages of the Health Vault Document Intelligence Pipeline.

### Key Principles:
- **Zero Mocks**: Every stage calls real Python `Container` handlers, PyMuPDF engine, Ollama models (`qwen3-vl:latest`, `bge-m3:latest`), and extraction services.
- **Granular Stage Telemetry**: Traces exact start/end timestamps, high-resolution duration, status, inputs, outputs, models, retries, and 8-class error classifications.
- **Per-Page OCR Deep-Dive**: Visualizes preprocessing time, OCR time, character/line counts, token limits, and identifies the slowest page.
- **Root-Cause Bottleneck Analysis**: Computes total wall-clock time, total measured stage time, subsystem breakdowns (OCR vs LLM vs Embeddings vs PDF Parsing), retry overhead, and duplicate processing detection.
- **Dual Telemetry Export**: Automatically exports full benchmark JSON and CSV telemetry reports.


In [ ]:
# ==============================================================================
# CELL 1: Environment Setup & Telemetry Tracer
# ==============================================================================
from __future__ import annotations
import sys, os, time, json, re, hashlib, traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Ensure ai-service root is in sys.path
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'scripts' else Path.cwd() / 'scripts'
AI_SERVICE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'scripts' else NOTEBOOK_DIR
if str(AI_SERVICE_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVICE_DIR))

import pandas as pd
import fitz  # PyMuPDF
from IPython.display import display, HTML, Markdown

class PipelineTracer:
    def __init__(self, doc_path: str | Path):
        self.doc_path = Path(doc_path)
        self.doc_name = self.doc_path.name
        self.stages: list[dict[str, Any]] = []
        self.ocr_pages: list[dict[str, Any]] = []
        self.start_wall_time = time.monotonic()
        self.start_iso = datetime.now(timezone.utc).isoformat()
        self.total_wall_sec = 0.0

    def now_iso(self) -> str:
        return datetime.now(timezone.utc).isoformat(timespec='milliseconds')

    def record_stage(
        self,
        stage_name: str,
        status: str,
        duration_sec: float,
        input_summary: str,
        output_summary: str,
        response_data: Any = None,
        error: Exception | None = None,
        model_info: dict[str, Any] | None = None,
        retries: int = 0,
        retry_timing_sec: float = 0.0,
    ) -> dict[str, Any]:
        err_dict = None
        err_class = None
        if error:
            err_dict = {
                'type': type(error).__name__,
                'message': str(error),
                'traceback': traceback.format_exc(),
            }
            err_class = self.classify_error(error)

        entry = {
            'stage': stage_name,
            'status': status,
            'started_at': self.now_iso(),
            'finished_at': self.now_iso(),
            'duration_sec': round(duration_sec, 3),
            'duration_ms': int(duration_sec * 1000),
            'input_summary': input_summary,
            'output_summary': output_summary,
            'response_data': response_data,
            'error': err_dict,
            'error_classification': err_class,
            'model_info': model_info or {},
            'retries': retries,
            'retry_timing_sec': round(retry_timing_sec, 3),
        }
        self.stages.append(entry)
        return entry

    def record_ocr_page(
        self,
        page_num: int,
        method: str,
        status: str,
        prep_duration_sec: float,
        ocr_duration_sec: float,
        chars_extracted: int,
        lines_count: int,
        confidence: float,
        raw_response: Any,
        parsed_response: Any,
        model_info: dict[str, Any] | None = None,
        tokens_info: dict[str, Any] | None = None,
        retries: int = 0,
        error: str | None = None,
    ) -> None:
        total_page_sec = prep_duration_sec + ocr_duration_sec
        self.ocr_pages.append({
            'page': page_num,
            'method': method,
            'status': status,
            'prep_duration_sec': round(prep_duration_sec, 3),
            'ocr_duration_sec': round(ocr_duration_sec, 3),
            'total_duration_sec': round(total_page_sec, 3),
            'chars_extracted': chars_extracted,
            'lines_count': lines_count,
            'confidence': round(confidence, 3),
            'raw_response': str(raw_response)[:500] if raw_response else '',
            'parsed_response': parsed_response,
            'model_info': model_info or {},
            'tokens_info': tokens_info or {},
            'retries': retries,
            'error': error,
            'is_slowest': False,
        })

    @staticmethod
    def classify_error(exc: Exception) -> str:
        msg = str(exc).lower()
        name = type(exc).__name__.lower()
        
        # 1. Programming errors (must never be classified as corrupt document)
        if any(sig in msg for sig in ('is not a function', 'attributeerror', 'typeerror', 'syntaxerror', 'nameerror', 'unboundlocalerror', 'keyerror', 'indexerror')) or name in ('typeerror', 'attributeerror', 'nameerror', 'syntaxerror', 'keyerror', 'indexerror'):
            return 'APPLICATION_ERROR'
            
        # 2. Network transport
        if any(sig in msg for sig in ('all connection attempts failed', 'connecterror', 'could not reach', 'eof occurred', 'incompleteread', 'connection reset', 'connection refused', 'remotedisconnected', 'transport')):
            return 'NETWORK_TRANSPORT_ERROR'
            
        # 3. Model timeout
        if 'timeout' in msg or 'timeout' in name or 'timed out' in msg:
            return 'MODEL_TIMEOUT'
            
        # 4. Model token limit
        if 'finish_reason' in msg or 'length' in msg or 'max_tokens' in msg or 'token limit' in msg or 'too many tokens' in msg:
            return 'MODEL_TOKEN_LIMIT'
            
        # 5. JSON parse failure
        if 'json' in msg or 'parse' in msg or 'ai_response_parse_failed' in msg or 'jsondecodeerror' in name:
            return 'JSON_PARSE_FAILURE'
            
        # 6. Parser incompatibility (unsupported PDF format, filter, encryption)
        if any(sig in msg for sig in ('incompatible', 'unsupported pdf', 'password required', 'unsupported filter', 'colorspace', 'unsupported colorspace', 'font unsupported')):
            return 'PARSER_INCOMPATIBILITY'
            
        # 7. Corrupt document (actual broken xref, zero bytes, damaged trailer)
        if any(sig in msg for sig in ('corrupt', 'cannot open broken', 'pdf syntax', 'broken xref', 'trailer missing', 'invalid header', 'empty payload')):
            return 'CORRUPT_DOCUMENT'
            
        # 8. Validation rejection (non-medical documents)
        if any(sig in msg for sig in ('nonmedical', 'non-medical', 'rejected as non-medical', 'utility bill', 'commercial receipt')):
            return 'VALIDATION_REJECTION'
            
        return 'APPLICATION_ERROR'

    def finish(self) -> None:
        self.total_wall_sec = round(time.monotonic() - self.start_wall_time, 3)

    def export_json(self, output_dir: Path | str = 'benchmark_results') -> Path:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        clean_stem = re.sub(r'[^a-zA-Z0-9_-]', '_', self.doc_path.stem)
        file_path = out_dir / f'benchmark_{clean_stem}_{ts}.json'
        
        payload = {
            'document': {
                'name': self.doc_name,
                'path': str(self.doc_path.resolve()),
                'size_bytes': self.doc_path.stat().st_size if self.doc_path.exists() else 0,
            },
            'timing': {
                'start_iso': self.start_iso,
                'finish_iso': self.now_iso(),
                'wall_clock_duration_sec': self.total_wall_sec,
                'sum_stages_duration_sec': round(sum(s['duration_sec'] for s in self.stages), 3),
            },
            'stages': self.stages,
            'ocr_pages': self.ocr_pages,
            'analysis': self.compute_analysis(),
        }
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(payload, f, indent=2, default=str, ensure_ascii=False)
        print(f'Exported comprehensive benchmark JSON to: {file_path.resolve()}')
        return file_path

    def export_csv(self, output_dir: Path | str = 'benchmark_results') -> tuple[Path, Path]:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        clean_stem = re.sub(r'[^a-zA-Z0-9_-]', '_', self.doc_path.stem)
        stages_csv = out_dir / f'benchmark_{clean_stem}_{ts}_stages.csv'
        pages_csv = out_dir / f'benchmark_{clean_stem}_{ts}_ocr_pages.csv'
        
        # Stages dataframe
        df_stages = pd.DataFrame([{
            'stage': s['stage'],
            'status': s['status'],
            'duration_sec': s['duration_sec'],
            'duration_ms': s['duration_ms'],
            'input_summary': s['input_summary'],
            'output_summary': s['output_summary'],
            'error_classification': s.get('error_classification') or '',
            'retries': s.get('retries', 0),
        } for s in self.stages])
        df_stages.to_csv(stages_csv, index=False, encoding='utf-8')
        
        # OCR pages dataframe
        if self.ocr_pages:
            df_pages = pd.DataFrame([{
                'page': p['page'],
                'method': p['method'],
                'status': p['status'],
                'prep_duration_sec': p['prep_duration_sec'],
                'ocr_duration_sec': p['ocr_duration_sec'],
                'total_duration_sec': p['total_duration_sec'],
                'chars_extracted': p['chars_extracted'],
                'lines_count': p['lines_count'],
                'confidence': p['confidence'],
                'retries': p['retries'],
                'error': p['error'] or '',
            } for p in self.ocr_pages])
            df_pages.to_csv(pages_csv, index=False, encoding='utf-8')
        else:
            with open(pages_csv, 'w', encoding='utf-8') as f:
                f.write('page,method,status,prep_duration_sec,ocr_duration_sec,total_duration_sec,chars_extracted,lines_count,confidence,retries,error\n')
        
        print(f'Exported benchmark CSVs to:\n  - {stages_csv.resolve()}\n  - {pages_csv.resolve()}')
        return stages_csv, pages_csv

    def compute_analysis(self) -> dict[str, Any]:
        total_stage_time = sum(s['duration_sec'] for s in self.stages) or 0.001
        stages_breakdown = []
        for s in self.stages:
            pct = round((s['duration_sec'] / total_stage_time) * 100, 1)
            stages_breakdown.append({
                'stage': s['stage'],
                'duration_sec': s['duration_sec'],
                'percent_of_total': pct,
                'status': s['status'],
            })

        slowest_stage = max(self.stages, key=lambda s: s['duration_sec'], default={'stage': 'None', 'duration_sec': 0})
        slowest_page = max(self.ocr_pages, key=lambda p: p['ocr_duration_sec'], default={'page': 0, 'ocr_duration_sec': 0, 'chars_extracted': 0})

        # Tag slowest page
        for p in self.ocr_pages:
            p['is_slowest'] = (p['page'] == slowest_page.get('page'))

        # Subsystem aggregation
        total_ocr_time = sum(s['duration_sec'] for s in self.stages if '07_page_ocr' in s['stage'] or '08_merge_ocr' in s['stage'])
        total_llm_time = sum(s['duration_sec'] for s in self.stages if '10_structured' in s['stage'] or 'summariz' in s['stage'] or 'clinical' in s['stage'])
        parsing_preprocessing_time = sum(s['duration_sec'] for s in self.stages if '03_pdf_validation' in s['stage'] or '06_page_preprocessing' in s['stage'] or '09_layout' in s['stage'])
        embedding_time = sum(s['duration_sec'] for s in self.stages if '12_embedding' in s['stage'])
        database_io_time = sum(s['duration_sec'] for s in self.stages if '02_file_read' in s['stage'] or '13_database_persistence' in s['stage'])
        retry_overhead_sec = sum(s.get('retry_timing_sec', 0.0) for s in self.stages)

        # Duplicate processing detection
        page_texts = [p.get('parsed_response') or '' for p in self.ocr_pages if isinstance(p.get('parsed_response'), str) and len(p.get('parsed_response', '')) > 20]
        duplicate_pages = len(page_texts) - len(set(page_texts))

        return {
            'total_wall_sec': self.total_wall_sec,
            'total_stage_sec': round(total_stage_time, 3),
            'stage_percentages': {s['stage']: round((s['duration_sec'] / total_stage_time) * 100, 1) for s in self.stages},
            'slowest_stage': {'name': slowest_stage['stage'], 'duration_sec': slowest_stage['duration_sec']},
            'slowest_ocr_page': {'page': slowest_page.get('page', 0), 'duration_sec': slowest_page.get('ocr_duration_sec', 0), 'chars': slowest_page.get('chars_extracted', 0)},
            'total_ocr_time': round(total_ocr_time, 3),
            'total_llm_time': round(total_llm_time, 3),
            'parsing_preprocessing_time': round(parsing_preprocessing_time, 3),
            'embedding_time': round(embedding_time, 3),
            'database_io_time': round(database_io_time, 3),
            'retry_overhead_sec': round(retry_overhead_sec, 3),
            'duplicate_processing_detected': duplicate_pages > 0,
            'duplicate_page_count': duplicate_pages,
            'stages_breakdown': stages_breakdown,
            'failed_stages': [s['stage'] for s in self.stages if s['status'] == 'FAILED'],
        }

print('PipelineTracer loaded successfully.')


In [ ]:
# ==============================================================================
# CELL 2: Initialize Real Production Container
# ==============================================================================
from app.settings import Settings
from app.container import Container

print('Initializing Health Vault Production Container...')
settings = Settings()
container = Container(settings)
try:
    await container.start()
    print('Production Container Initialized successfully.')
except Exception as exc:
    print(f'Note: Remote model warm-up encountered network exception: {exc}')
    print('Production Container loaded in offline/degraded mode. Local stages will execute fully; remote model stages will record transport status.')
print(f'  AI Model (Vision):    {settings.ai_model}')
print(f'  Chat Model:           {settings.chat_model}')
print(f'  Embedding Model:      {settings.ai_embedding_model}')
print(f'  AI Base URL:          {settings.ai_base_url}')
print(f'  AI Timeout:           {settings.ai_timeout_seconds}s')
print(f'  Max Output Tokens:    {settings.ai_max_output_tokens}')
print(f'  Page Concurrency:     {settings.ai_page_concurrency}')


In [ ]:
# ==============================================================================
# CELL 3: Document Discovery & Selection
# ==============================================================================
DEFAULT_REPORTS_DIR = Path(r"D:\TECHROVER\PROJECTS\RTH\healh-vault-project\Medical Reports\Masa's Report")
FALLBACK_DIR = AI_SERVICE_DIR / 'tests' / 'fixtures' / 'golden_docs'
SUPPORTED_EXTS = {'.pdf', '.png', '.jpg', '.jpeg', '.webp', '.tiff', '.bmp'}

def discover_documents(reports_dir: Path | str = DEFAULT_REPORTS_DIR) -> list[Path]:
    p = Path(reports_dir)
    search_dir = p if p.exists() and p.is_dir() else FALLBACK_DIR
    if not search_dir.exists():
        return []
    return sorted([f for f in search_dir.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

available_docs = discover_documents()
print(f'Discovered {len(available_docs)} document(s):')
for idx, d in enumerate(available_docs, 1):
    sz_kb = round(d.stat().st_size / 1024, 1)
    print(f'  {idx:2d}. {d.name:<35} ({sz_kb} KB)')

# ---- CHANGE THIS TO BENCHMARK ANY SPECIFIC DOCUMENT ----
# e.g. 'Manjuben Ranoliya.pdf', '5069200904.pdf', 'P1(CIMS)(21042022).pdf'
SELECTED_DOCUMENT_NAME = '5069200904.pdf'

# Resolve path
matched = [d for d in available_docs if d.name.lower() == SELECTED_DOCUMENT_NAME.lower()]
SELECTED_DOCUMENT_PATH = matched[0] if matched else (available_docs[0] if available_docs else None)

if SELECTED_DOCUMENT_PATH:
    print(f'\nActive Target Document: {SELECTED_DOCUMENT_PATH.name}')
    print(f'  Full Path: {SELECTED_DOCUMENT_PATH.resolve()}')
    print(f'  Size:      {round(SELECTED_DOCUMENT_PATH.stat().st_size/1024, 1)} KB')
else:
    print('Warning: No document selected or directory not found.')


In [ ]:
# ==============================================================================
# STAGE 1 & 2: Document Selection & File Read / Metadata
# ==============================================================================
def stage_01_select_document(path: Path | str, tracer: PipelineTracer) -> dict[str, Any]:
    t0 = time.perf_counter()
    p = Path(path)
    if not p.exists():
        err = FileNotFoundError(f'Document not found: {p}')
        tracer.record_stage('01_select_document', 'FAILED', time.perf_counter() - t0, str(p), 'File missing', error=err)
        raise err
    res = {
        'filename': p.name,
        'extension': p.suffix.lower(),
        'absolute_path': str(p.resolve()),
        'exists': True,
    }
    tracer.record_stage('01_select_document', 'SUCCESS', time.perf_counter() - t0, str(p), f'Selected {p.name}', res)
    return res

def stage_02_file_read(path: Path | str, tracer: PipelineTracer) -> tuple[dict[str, Any], bytes]:
    t0 = time.perf_counter()
    p = Path(path)
    data = p.read_bytes()
    sha256_hash = hashlib.sha256(data).hexdigest()
    header_hex = data[:16].hex()
    res = {
        'filename': p.name,
        'size_bytes': len(data),
        'size_kb': round(len(data) / 1024, 2),
        'size_mb': round(len(data) / (1024 * 1024), 2),
        'sha256': sha256_hash,
        'magic_header_hex': header_hex,
        'is_pdf_signature': data.startswith(b'%PDF'),
    }
    tracer.record_stage('02_file_read', 'SUCCESS', time.perf_counter() - t0, f'File: {p.name}', f'{res["size_kb"]} KB read, SHA256: {sha256_hash[:12]}...', res)
    return res, data


In [ ]:
# ==============================================================================
# STAGE 3 & 4: PDF PyMuPDF Validation & Pre-Upload Medical Gate
# ==============================================================================
def stage_03_pdf_validation(file_bytes: bytes, filename: str, tracer: PipelineTracer) -> tuple[dict[str, Any], fitz.Document | None]:
    t0 = time.perf_counter()
    if not filename.lower().endswith('.pdf'):
        res = {'is_pdf': False, 'message': 'Raster image; PDF structural validation bypassed'}
        tracer.record_stage('03_pdf_validation', 'SKIPPED', time.perf_counter() - t0, filename, 'Raster image', res)
        return res, None
    try:
        doc = fitz.open(stream=file_bytes, filetype='pdf')
        page_count = len(doc)
        per_page_text_len = []
        total_direct_chars = 0
        for p in doc:
            txt = p.get_text()
            per_page_text_len.append(len(txt))
            total_direct_chars += len(txt)
        
        is_scanned = total_direct_chars < 50 * max(1, page_count)
        res = {
            'is_pdf': True,
            'page_count': page_count,
            'is_encrypted': doc.is_encrypted,
            'total_direct_chars': total_direct_chars,
            'is_scanned': is_scanned,
            'per_page_chars': per_page_text_len,
            'routing': 'Scanned OCR Fallback' if is_scanned else 'Born-Digital Direct Fast-Path',
        }
        tracer.record_stage('03_pdf_validation', 'SUCCESS', time.perf_counter() - t0, f'{page_count} pages', f'{res["routing"]} ({total_direct_chars} chars)', res)
        return res, doc
    except Exception as exc:
        tracer.record_stage('03_pdf_validation', 'FAILED', time.perf_counter() - t0, filename, 'PDF parse error', error=exc)
        raise

async def stage_04_preupload_validation(container: Container, file_bytes: bytes, filename: str, tracer: PipelineTracer) -> dict[str, Any]:
    t0 = time.perf_counter()
    try:
        res = await container.ocr_stage_handler.validate_document(file_bytes=file_bytes, filename=filename)
        is_valid = res.get('isValid', False)
        is_med = res.get('isMedical', False)
        conf = res.get('confidence', 0.0)
        status = 'SUCCESS' if is_valid else 'FAILED'
        tracer.record_stage('04_preupload_validation', status, time.perf_counter() - t0, f'{len(file_bytes)} bytes', f'Valid={is_valid}, Medical={is_med}, Conf={conf}', res)
        return res
    except Exception as exc:
        tracer.record_stage('04_preupload_validation', 'FAILED', time.perf_counter() - t0, filename, 'Validation exception', error=exc)
        raise


In [ ]:
# ==============================================================================
# STAGE 5 & 6: Medical Classification & Image Preprocessing
# ==============================================================================
from app.services.pipeline.preprocessing import preprocess_document_image

def stage_05_medical_classification(validation_res: dict[str, Any], tracer: PipelineTracer) -> dict[str, Any]:
    t0 = time.perf_counter()
    is_medical = validation_res.get('isMedical', True)
    confidence = validation_res.get('confidence', 1.0)
    fail_open = validation_res.get('failOpenUsed', False)
    classification = 'MEDICAL_DOCUMENT' if is_medical else 'NON_MEDICAL_DOCUMENT'
    res = {
        'classification': classification,
        'is_medical': is_medical,
        'confidence': confidence,
        'fail_open_used': fail_open,
    }
    status = 'SUCCESS' if is_medical else 'FAILED'
    tracer.record_stage('05_medical_classification', status, time.perf_counter() - t0, f'Gate: {classification}', f'{classification} (conf: {confidence})', res)
    if not is_medical:
        raise ValueError(f'Document rejected as non-medical: {validation_res}')
    return res

def stage_06_page_preprocessing(doc: fitz.Document | None, file_bytes: bytes, filename: str, is_scanned: bool, tracer: PipelineTracer) -> list[dict[str, Any]]:
    t0 = time.perf_counter()
    preprocessed_pages = []
    try:
        if doc is not None and len(doc) > 0:
            for idx, page in enumerate(doc, start=1):
                t_p = time.perf_counter()
                pix = page.get_pixmap(dpi=150)
                raw_png = pix.tobytes('png')
                prep_bytes = preprocess_document_image(raw_png, deskew=True, remove_shadows=True, binarize=False)
                dur_p = time.perf_counter() - t_p
                preprocessed_pages.append({
                    'page_num': idx,
                    'width': pix.width,
                    'height': pix.height,
                    'raw_bytes_len': len(raw_png),
                    'preprocessed_bytes': prep_bytes,
                    'preprocessed_bytes_len': len(prep_bytes),
                    'preprocessing_duration_sec': round(dur_p, 3),
                })
        else:
            t_p = time.perf_counter()
            prep_bytes = preprocess_document_image(file_bytes, deskew=True, remove_shadows=True, binarize=False)
            dur_p = time.perf_counter() - t_p
            preprocessed_pages.append({
                'page_num': 1,
                'width': 0,
                'height': 0,
                'raw_bytes_len': len(file_bytes),
                'preprocessed_bytes': prep_bytes,
                'preprocessed_bytes_len': len(prep_bytes),
                'preprocessing_duration_sec': round(dur_p, 3),
            })
        total_dur = time.perf_counter() - t0
        tracer.record_stage('06_page_preprocessing', 'SUCCESS', total_dur, f'{len(preprocessed_pages)} pages', f'{len(preprocessed_pages)} page(s) deskewed & shadows removed', [{'page': p['page_num'], 'raw_bytes': p['raw_bytes_len'], 'prep_bytes': p['preprocessed_bytes_len'], 'prep_time_s': p['preprocessing_duration_sec']} for p in preprocessed_pages])
        return preprocessed_pages
    except Exception as exc:
        tracer.record_stage('06_page_preprocessing', 'FAILED', time.perf_counter() - t0, filename, 'Preprocessing exception', error=exc)
        raise


In [ ]:
# ==============================================================================
# STAGE 7 & 8: Page-wise OCR / Vision Extraction & Result Merge
# ==============================================================================
async def stage_07_page_ocr_extraction(
    container: Container,
    preprocessed_pages: list[dict[str, Any]],
    doc: fitz.Document | None,
    is_scanned: bool,
    tracer: PipelineTracer,
) -> list[dict[str, Any]]:
    t0 = time.perf_counter()
    page_ocr_results = []
    min_chars = getattr(container.settings, 'ai_min_text_chars', 50)

    for p_info in preprocessed_pages:
        p_num = p_info['page_num']
        prep_bytes = p_info['preprocessed_bytes']
        prep_dur = p_info['preprocessing_duration_sec']
        t_page_start = time.perf_counter()
        
        # Check if direct born-digital text is available on this specific page
        direct_text = ''
        if doc and p_num <= len(doc):
            direct_text = doc[p_num - 1].get_text() or ''
        
        if len(direct_text.strip()) >= min_chars and not is_scanned:
            # PyMuPDF fast-path
            ocr_dur = time.perf_counter() - t_page_start
            lines = [{'text': l.strip(), 'confidence': 1.0} for l in direct_text.splitlines() if l.strip()]
            res = {
                'page': p_num,
                'text': direct_text,
                'confidence': 1.0,
                'lines': lines,
                'method': 'PyMuPDF Direct Text',
                'preprocessing_duration_sec': prep_dur,
                'ocr_duration_sec': round(ocr_dur, 3),
                'error': None,
            }
            page_ocr_results.append(res)
            tracer.record_ocr_page(
                page_num=p_num,
                method='PyMuPDF Direct Text',
                status='SUCCESS',
                prep_duration_sec=prep_dur,
                ocr_duration_sec=ocr_dur,
                chars_extracted=len(direct_text),
                lines_count=len(lines),
                confidence=1.0,
                raw_response=direct_text[:500],
                parsed_response=direct_text,
                model_info={'engine': 'PyMuPDF', 'model': 'fitz_native'},
                retries=0,
            )
            print(f'   [Page {p_num}] PyMuPDF Fast-Path -> {len(direct_text)} chars in {ocr_dur:.3f}s (prep: {prep_dur:.3f}s)')
        else:
            # Qwen Vision Extraction via real container.vision
            model_name = getattr(container.settings, 'ai_model', 'qwen3-vl:latest')
            try:
                ocr_res = await container.vision.extract_image(
                    prep_bytes,
                    filename=f'page_{p_num}.png',
                    mime_type='image/png',
                    max_pages=1,
                )
                ocr_dur = time.perf_counter() - t_page_start
                extracted_text = ocr_res.get('text') or ''
                lines = [{'text': l.strip(), 'confidence': 0.95} for l in extracted_text.splitlines() if l.strip()]
                metrics = ocr_res.get('metrics', {})
                res = {
                    'page': p_num,
                    'text': extracted_text,
                    'confidence': 0.95,
                    'lines': lines,
                    'method': f'Vision Model ({model_name})',
                    'preprocessing_duration_sec': prep_dur,
                    'ocr_duration_sec': round(ocr_dur, 3),
                    'error': None,
                }
                page_ocr_results.append(res)
                tracer.record_ocr_page(
                    page_num=p_num,
                    method=f'Vision ({model_name})',
                    status='SUCCESS',
                    prep_duration_sec=prep_dur,
                    ocr_duration_sec=ocr_dur,
                    chars_extracted=len(extracted_text),
                    lines_count=len(lines),
                    confidence=0.95,
                    raw_response=ocr_res,
                    parsed_response=extracted_text,
                    model_info={'engine': 'Ollama/Vision', 'model': model_name},
                    tokens_info={'output_tokens': metrics.get('output_tokens'), 'truncated': metrics.get('truncated', False)},
                    retries=0,
                )
                print(f'   [Page {p_num}] Qwen Vision -> {len(extracted_text)} chars in {ocr_dur:.3f}s (prep: {prep_dur:.3f}s)')
            except Exception as exc:
                ocr_dur = time.perf_counter() - t_page_start
                err_str = str(exc)
                res = {
                    'page': p_num,
                    'text': '',
                    'confidence': 0.0,
                    'lines': [],
                    'method': f'Vision Model ({model_name})',
                    'preprocessing_duration_sec': prep_dur,
                    'ocr_duration_sec': round(ocr_dur, 3),
                    'error': err_str,
                }
                page_ocr_results.append(res)
                tracer.record_ocr_page(
                    page_num=p_num,
                    method=f'Vision ({model_name})',
                    status='FAILED',
                    prep_duration_sec=prep_dur,
                    ocr_duration_sec=ocr_dur,
                    chars_extracted=0,
                    lines_count=0,
                    confidence=0.0,
                    raw_response=None,
                    parsed_response='',
                    model_info={'engine': 'Ollama/Vision', 'model': model_name},
                    retries=0,
                    error=err_str,
                )
                print(f'   [Page {p_num}] Vision OCR FAILED -> {err_str}')

    total_dur = time.perf_counter() - t0
    total_chars = sum(len(p['text']) for p in page_ocr_results)
    tracer.record_stage('07_page_ocr_extraction', 'SUCCESS', total_dur, f'{len(preprocessed_pages)} pages', f'{total_chars} total chars extracted across {len(page_ocr_results)} page(s)', page_ocr_results)
    return page_ocr_results

def stage_08_merge_ocr_results(page_results: list[dict[str, Any]], tracer: PipelineTracer) -> dict[str, Any]:
    t0 = time.perf_counter()
    full_text_parts = [p['text'] for p in page_results if p.get('text')]
    merged_text = '\n\n'.join(full_text_parts)
    total_chars = len(merged_text)
    
    # Paragraph reconstruction
    paragraphs = []
    for p in page_results:
        for para in p['text'].split('\n\n'):
            if para.strip():
                paragraphs.append({'page': p['page'], 'text': para.strip()})
    
    raw_ocr_data = {
        'pages': page_results,
        'fullText': merged_text,
        'text': merged_text,
        'paragraphs': paragraphs,
        'pageCount': len(page_results),
        'characterCount': total_chars,
        'detectedLanguages': ['english'],
        'confidence': 1.0 if all(p.get('confidence', 0) > 0.9 for p in page_results) else 0.8,
        'metrics': {
            'page_count': len(page_results),
            'total_characters': total_chars,
            'elapsed_ms': int((time.perf_counter() - t0) * 1000),
        },
    }
    tracer.record_stage('08_merge_ocr_results', 'SUCCESS', time.perf_counter() - t0, f'{len(page_results)} page result(s)', f'Canonical raw_ocr_data compiled ({total_chars} chars, {len(paragraphs)} paras)', raw_ocr_data)
    return raw_ocr_data


In [ ]:
# ==============================================================================
# STAGE 9 & 10: Layout Hierarchy & Structured Medical Data Extraction
# ==============================================================================
def stage_09_layout_parsing(container: Container, raw_ocr_data: dict[str, Any], tracer: PipelineTracer) -> dict[str, Any]:
    t0 = time.perf_counter()
    try:
        layout_data = container.layout_stage_handler.parse_layout(raw_ocr_data)
        sections = [s.get('title') for s in layout_data.get('sections', [])]
        tables = layout_data.get('tables', [])
        summary = f'{len(sections)} sections ({sections[:4]}), {len(tables)} table(s)'
        tracer.record_stage('09_layout_parsing', 'SUCCESS', time.perf_counter() - t0, f'{raw_ocr_data["characterCount"]} chars', summary, layout_data)
        return layout_data
    except Exception as exc:
        tracer.record_stage('09_layout_parsing', 'FAILED', time.perf_counter() - t0, 'OCR text input', 'Layout decomposition error', error=exc)
        raise

async def stage_10_structured_extraction(container: Container, raw_ocr_data: dict[str, Any], layout_data: dict[str, Any], tracer: PipelineTracer) -> dict[str, Any]:
    t0 = time.perf_counter()
    try:
        structured = await container.clinical_stage_handler.extract_fields(raw_ocr_data, layout_data)
        pt = structured.get('patientInfo', {})
        meds = structured.get('medications', [])
        labs = structured.get('labResults', [])
        diags = structured.get('diagnosis', [])
        summary = f'Patient: {pt.get("name", "N/A")}, {len(diags)} diagnoses, {len(meds)} meds, {len(labs)} labs'
        tracer.record_stage('10_structured_extraction', 'SUCCESS', time.perf_counter() - t0, f'{raw_ocr_data["characterCount"]} chars', summary, structured)
        return structured
    except Exception as exc:
        tracer.record_stage('10_structured_extraction', 'FAILED', time.perf_counter() - t0, 'Clinical text', 'Field extraction error', error=exc)
        raise


In [ ]:
# ==============================================================================
# STAGE 11 & 12: Visual Graph Asset Extraction & Vector Embeddings
# ==============================================================================
async def stage_11_graph_extraction(container: Container, file_path: Path, raw_ocr_data: dict[str, Any], tracer: PipelineTracer) -> list[dict[str, Any]]:
    t0 = time.perf_counter()
    try:
        bucket = getattr(container.settings, 'patient_documents_bucket', 'patient-documents')
        file_key = f'benchmark/{file_path.name}'
        graphs = []
        if file_path.suffix.lower() == '.pdf':
            graphs = await asyncio.to_thread(
                container.graph_stage_handler._extract_pdf_visual_assets, file_path, file_key, bucket
            )
        else:
            graphs = await asyncio.to_thread(
                container.graph_stage_handler._extract_raster_visual_assets, file_path, file_key, bucket
            )
        summary = f'{len(graphs)} visual crop(s) extracted (charts, ECGs, thumbnails)'
        tracer.record_stage('11_graph_extraction', 'SUCCESS', time.perf_counter() - t0, str(file_path.name), summary, graphs)
        return graphs
    except Exception as exc:
        tracer.record_stage('11_graph_extraction', 'FAILED', time.perf_counter() - t0, str(file_path.name), 'Graph extraction error', error=exc)
        return []

async def stage_12_embedding_generation(container: Container, full_text: str, tracer: PipelineTracer) -> dict[str, Any]:
    t0 = time.perf_counter()
    try:
        embeddings = await container.embedding_stage_handler.generate_embeddings(full_text)
        chunk_count = embeddings.get('chunkCount', len(embeddings.get('chunks', [])))
        dim = embeddings.get('vectorDimension', 1024)
        summary = f'{chunk_count} chunk(s) vectorized with dimension {dim} (BGE-M3)'
        tracer.record_stage('12_embedding_generation', 'SUCCESS', time.perf_counter() - t0, f'{len(full_text)} chars', summary, embeddings)
        return embeddings
    except Exception as exc:
        tracer.record_stage('12_embedding_generation', 'FAILED', time.perf_counter() - t0, f'{len(full_text)} chars', 'Embedding error', error=exc)
        raise


In [ ]:
# ==============================================================================
# STAGE 13 & 14: Persistence Serialization & Final Response Assembly
# ==============================================================================
import uuid
from uuid import UUID

async def stage_13_database_persistence(
    container: Container,
    structured_data: dict[str, Any],
    raw_ocr_data: dict[str, Any],
    file_name: str,
    tracer: PipelineTracer,
    dry_run: bool = True,
) -> dict[str, Any]:
    t0 = time.perf_counter()
    job_id = uuid.uuid4()
    checkpoint_payload = {
        'jobId': str(job_id),
        'filename': file_name,
        'extractedEntities': len(structured_data.get('medications', [])) + len(structured_data.get('labResults', [])),
        'isDryRun': dry_run,
        'serializable': True,
    }
    try:
        # Verify full serializability into JSON string without exceptions
        json_str = json.dumps(structured_data, default=str)
        checkpoint_payload['serialized_bytes'] = len(json_str)
        summary = f'Checkpoint verified (DryRun={dry_run}, {len(json_str)} JSON bytes)'
        tracer.record_stage('13_database_persistence', 'SUCCESS', time.perf_counter() - t0, f'Job: {job_id}', summary, checkpoint_payload)
        return checkpoint_payload
    except Exception as exc:
        tracer.record_stage('13_database_persistence', 'FAILED', time.perf_counter() - t0, f'Job: {job_id}', 'Persistence error', error=exc)
        raise

def stage_14_final_response(
    tracer: PipelineTracer,
    raw_ocr_data: dict[str, Any],
    structured_data: dict[str, Any],
    layout_data: dict[str, Any],
    embedding_data: dict[str, Any],
) -> dict[str, Any]:
    t0 = time.perf_counter()
    tracer.finish()
    final_payload = {
        'status': 'SUCCESS' if not any(s['status'] == 'FAILED' for s in tracer.stages) else 'FAILED',
        'document': {
            'name': tracer.doc_name,
            'pageCount': raw_ocr_data.get('pageCount', 1),
            'characterCount': raw_ocr_data.get('characterCount', 0),
        },
        'structuredData': structured_data,
        'layoutSummary': {
            'sections': [s.get('title') for s in layout_data.get('sections', [])],
            'tableCount': len(layout_data.get('tables', [])),
        },
        'embeddingsSummary': {
            'chunkCount': embedding_data.get('chunkCount', len(embedding_data.get('chunks', []))),
            'vectorDimension': embedding_data.get('vectorDimension', 1024),
        },
        'telemetry': {
            'totalWallTimeSec': tracer.total_wall_sec,
            'stageCount': len(tracer.stages),
            'failedStages': [s['stage'] for s in tracer.stages if s['status'] == 'FAILED'],
        },
    }
    tracer.record_stage('14_final_response', 'SUCCESS', time.perf_counter() - t0, 'All pipeline artifacts', f'Final response assembled in {tracer.total_wall_sec}s', final_payload)
    return final_payload


In [ ]:
# ==============================================================================
# MASTER EXECUTION FUNCTION: Run All 14 Stages Sequentially
# ==============================================================================
async def run_full_pipeline(doc_path: Path | str, dry_run_db: bool = True) -> tuple[PipelineTracer, dict[str, Any]]:
    tracer = PipelineTracer(doc_path)
    print(f'\n======================================================================')
    print(f'Starting Granular 14-Stage Trace for: {tracer.doc_name}')
    print(f'======================================================================')
    
    # Stage 1: Select
    stage_01_select_document(doc_path, tracer)
    
    # Stage 2: Read
    meta, file_bytes = stage_02_file_read(doc_path, tracer)
    
    # Stage 3: PDF validation
    pdf_info, doc = stage_03_pdf_validation(file_bytes, tracer.doc_name, tracer)
    is_scanned = pdf_info.get('is_scanned', True)
    
    # Stage 4: Pre-upload validation
    val_res = await stage_04_preupload_validation(container, file_bytes, tracer.doc_name, tracer)
    
    # Stage 5: Medical classification
    stage_05_medical_classification(val_res, tracer)
    
    # Stage 6: Preprocessing
    preprocessed_pages = stage_06_page_preprocessing(doc, file_bytes, tracer.doc_name, is_scanned, tracer)
    
    # Stage 7: Page-wise OCR / Vision
    ocr_pages = await stage_07_page_ocr_extraction(container, preprocessed_pages, doc, is_scanned, tracer)
    
    # Stage 8: Merge OCR results
    raw_ocr = stage_08_merge_ocr_results(ocr_pages, tracer)
    
    # Stage 9: Layout parsing
    layout = stage_09_layout_parsing(container, raw_ocr, tracer)
    
    # Stage 10: Structured extraction
    structured = await stage_10_structured_extraction(container, raw_ocr, layout, tracer)
    
    # Stage 11: Graph visual assets
    graphs = await stage_11_graph_extraction(container, Path(doc_path), raw_ocr, tracer)
    
    # Stage 12: Embeddings
    embed = await stage_12_embedding_generation(container, raw_ocr['fullText'], tracer)
    
    # Stage 13: Persistence
    await stage_13_database_persistence(container, structured, raw_ocr, tracer.doc_name, tracer, dry_run=dry_run_db)
    
    # Stage 14: Final response
    final_resp = stage_14_final_response(tracer, raw_ocr, structured, layout, embed)
    
    return tracer, final_resp


In [ ]:
# ==============================================================================
# CELL: Execute Full Benchmark Pipeline
# ==============================================================================
if SELECTED_DOCUMENT_PATH:
    tracer, final_response = await run_full_pipeline(SELECTED_DOCUMENT_PATH, dry_run_db=True)
    print(f'\nPipeline Finished. Total Wall Time: {tracer.total_wall_sec}s')
else:
    print('Error: No target document selected.')


In [ ]:
# ==============================================================================
# CELL: Stage-by-Stage Results Table (Format: Stage | Status | Duration | Output Summary)
# ==============================================================================
table_rows = []
for s in tracer.stages:
    status_str = s['status']
    if s.get('error_classification'):
        status_str += f" ({s['error_classification']})"
    table_rows.append({
        'Stage': s['stage'],
        'Status': status_str,
        'Duration': f"{s['duration_sec']:.3f}s",
        'Output Summary': s['output_summary'],
    })

df_stages = pd.DataFrame(table_rows)
display(HTML('<h3>Pipeline Execution Stage Summary</h3>'))
display(HTML(df_stages.to_html(index=False, classes='table table-striped table-bordered')))


In [ ]:
# ==============================================================================
# CELL: Per-Page OCR Diagnostics
# ==============================================================================
if tracer.ocr_pages:
    ocr_table_rows = []
    for p in tracer.ocr_pages:
        ocr_table_rows.append({
            'Page': p['page'],
            'Method': p['method'],
            'Status': p['status'],
            'Prep (s)': f"{p['prep_duration_sec']:.3f}s",
            'OCR (s)': f"{p['ocr_duration_sec']:.3f}s",
            'Total (s)': f"{p['total_duration_sec']:.3f}s",
            'Chars': p['chars_extracted'],
            'Lines': p['lines_count'],
            'Confidence': p['confidence'],
            'Slowest?': '⭐ SLOWEST' if p.get('is_slowest') else '',
            'Error': p['error'] or '',
        })
    df_ocr = pd.DataFrame(ocr_table_rows)
    display(HTML('<h3>Per-Page OCR Telemetry</h3>'))
    display(HTML(df_ocr.to_html(index=False, classes='table table-hover table-bordered')))
else:
    print('No per-page OCR telemetry recorded.')


In [ ]:
# ==============================================================================
# CELL: Expandable Intermediate Output Inspector
# ==============================================================================
html_cards = ['<h3>Intermediate Stage Output Inspector</h3>']
for s in tracer.stages:
    st_name = s['stage']
    st_status = s['status']
    badge_color = '#28a745' if st_status == 'SUCCESS' else ('#ffc107' if st_status == 'SKIPPED' else '#dc3545')
    json_preview = json.dumps(s.get('response_data'), indent=2, default=str) if s.get('response_data') else 'None'
    err_preview = json.dumps(s.get('error'), indent=2) if s.get('error') else 'None'
    
    card = f'''
    <details style="margin-bottom: 8px; border: 1px solid #ddd; border-radius: 6px; padding: 8px; background: #fafafa;">
      <summary style="cursor: pointer; font-weight: bold;">
        <span style="background: {badge_color}; color: white; padding: 2px 8px; border-radius: 4px; font-size: 11px;">{st_status}</span>
        {st_name} &mdash; {s['duration_sec']}s &mdash; <span style="color: #666; font-weight: normal;">{s['output_summary']}</span>
      </summary>
      <div style="padding: 10px 0 0 10px;">
        <p><b>Input:</b> {s['input_summary']}</p>
        <p><b>Duration:</b> {s['duration_ms']} ms ({s['duration_sec']} sec)</p>
        <details>
          <summary style="cursor: pointer; color: #0366d6;">View Intermediate Response JSON ({len(json_preview)} chars)</summary>
          <pre style="background: #272822; color: #f8f8f2; padding: 10px; border-radius: 4px; max-height: 350px; overflow-y: auto;">{json_preview}</pre>
        </details>'''
    if s.get('error'):
        card += f'''
        <div style="margin-top: 8px; color: #721c24; background: #f8d7da; border: 1px solid #f5c6cb; padding: 8px; border-radius: 4px;">
          <b>Error Classification:</b> {s.get('error_classification')}<br/>
          <pre style="margin: 4px 0 0 0;">{err_preview}</pre>
        </div>'''
    card += '</div></details>'
    html_cards.append(card)

display(HTML(''.join(html_cards)))


In [ ]:
# ==============================================================================
# CELL: Extracted Clinical Findings Display
# ==============================================================================
structured_data = next((s['response_data'] for s in tracer.stages if s['stage'] == '10_structured_extraction'), {})

if structured_data and isinstance(structured_data, dict):
    pt = structured_data.get('patientInfo', {})
    doc = structured_data.get('doctorInfo', {})
    hosp = structured_data.get('hospitalInfo', {})
    diags = structured_data.get('diagnosis', [])
    meds = structured_data.get('medications', [])
    labs = structured_data.get('labResults', [])
    vitals = structured_data.get('vitals', [])
    
    display(HTML('<h3>Final Extracted Medical Data</h3>'))
    print(f'Patient Info:    {pt}')
    print(f'Doctor Info:     {doc}')
    print(f'Hospital Info:   {hosp}')
    print(f'Diagnoses:       {diags}')
    print(f'Vitals:          {vitals}')
    
    if meds:
        display(HTML('<h4>Medications</h4>'))
        display(pd.DataFrame(meds))
    
    if labs:
        display(HTML('<h4>Lab Results & Reference Ranges (In-Code Evaluated)</h4>'))
        display(pd.DataFrame(labs))
else:
    print('No structured medical data extracted.')


In [ ]:
# ==============================================================================
# CELL: Bottleneck & Root Cause Analysis
# ==============================================================================
analysis = tracer.compute_analysis()

display(HTML('<h3>Root Cause & Bottleneck Report</h3>'))
print(f"Total Pipeline Wall Time:         {analysis['total_wall_sec']:.3f}s")
print(f"Total Measured Stage Time:        {analysis['total_stage_sec']:.3f}s")
slow_pct = analysis['stage_percentages'].get(analysis['slowest_stage']['name'], 0.0)
print(f"Slowest Overall Stage:            {analysis['slowest_stage']['name']} ({analysis['slowest_stage']['duration_sec']}s, {slow_pct}%)")
if analysis['slowest_ocr_page']['page'] > 0:
    print(f"Slowest OCR Page:                 Page {analysis['slowest_ocr_page']['page']} ({analysis['slowest_ocr_page']['duration_sec']}s, {analysis['slowest_ocr_page']['chars']} chars)")

print('\nSubsystem Time Consumption & Percentages:')
subsystems = [
    ('Total OCR Time', analysis['total_ocr_time']),
    ('Total Ollama/LLM Time', analysis['total_llm_time']),
    ('Parsing & Preprocessing', analysis['parsing_preprocessing_time']),
    ('Embedding Time', analysis['embedding_time']),
    ('Database & I/O Time', analysis['database_io_time']),
    ('Retry Overhead', analysis['retry_overhead_sec']),
]
for label, dur in subsystems:
    pct = round((dur / max(0.001, analysis['total_stage_sec'])) * 100, 1)
    print(f"  - {label:<25}: {dur:>7.3f}s ({pct:>5.1f}%)")

print(f"\nDuplicate Processing Detected:     {analysis['duplicate_processing_detected']} ({analysis['duplicate_page_count']} duplicate pages)")

# Automated Diagnostic Analysis
print('\nDiagnostic Findings & Recommendations:')
slow_name = analysis['slowest_stage']['name']
slow_dur = analysis['slowest_stage']['duration_sec']
failed = analysis['failed_stages']

if failed:
    print(f'  [CRITICAL] {len(failed)} stage(s) FAILED: {failed}. Check error traces above for root causes.')
else:
    print('  [SUCCESS] All pipeline stages executed successfully with zero fatal exceptions.')

if 'OCR' in slow_name or 'Vision' in slow_name:
    print(f'  [BOTTLENECK] Stage {slow_name} dominates execution time ({slow_dur}s).')
    print('               Scanned pages require remote vision LLM inference. For documents with born-digital text, ensure PyMuPDF fast-path is active.')
    print('               For high-volume scanned PDFs, consider parallel page batching or smaller DPI rendering (e.g. DPI=120).')
elif 'Structured' in slow_name:
    print(f'  [BOTTLENECK] Structured Clinical Extraction is the slowest stage ({slow_dur}s).')
    print('               LLM normalization prompt is parsing full document text. Verify that token predictions are bounded.')
elif 'Embedding' in slow_name:
    print(f'  [BOTTLENECK] BGE-M3 Dense Embedding is the slowest stage ({slow_dur}s).')
    print('               Ensure embeddings use batch arrays rather than serial single-chunk HTTP requests.')


In [ ]:
# ==============================================================================
# CELL: Export Telemetry JSON & CSV
# ==============================================================================
export_json_path = tracer.export_json(NOTEBOOK_DIR / 'benchmark_results')
stages_csv_path, pages_csv_path = tracer.export_csv(NOTEBOOK_DIR / 'benchmark_results')
print(f'Saved Telemetry Artifacts:')
print(f'  JSON:       {export_json_path.name}')
print(f'  Stages CSV: {stages_csv_path.name}')
print(f'  Pages CSV:  {pages_csv_path.name}')
